# 03 — Scanning / PF indexing (voxel grid)

The same unified indexer drives **PF-HEDM / scanning** workflows: it
sweeps a grid of sample voxels, and for each voxel runs the seed
pipeline with the scan-position beam filter active. Output is the
consolidated `IndexBest_all.bin` (one block of records per voxel),
readable by the `pf_MIDAS` downstream.

A full scanning fixture is heavy (hundreds of voxels, ~GB bins), so
this notebook builds a **tiny synthetic in-memory case** — a 3×3
voxel grid with minimal observations — to exercise the
`run_scanning` orchestration + the consolidated reader end-to-end in
a couple of seconds on CPU.


In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # torch+numba libomp coexist
import tempfile
from pathlib import Path

import numpy as np
import torch

from midas_index import Indexer
from midas_index.params import IndexerParams
from midas_index.io.consolidated import read_index_best_all

WORK = Path(tempfile.mkdtemp(prefix="midas_index_scan_nb_"))
print("workspace:", WORK)


workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_index_scan_nb_u_s1kloc


## 1. Minimal scanning params + observations

We construct `IndexerParams` directly (no file) with the scan filter
enabled (`scan_pos_tol_um`, single-sided Friedel — the C default) and
hand the indexer a minimal observation set so it doesn't read any
files. The voxel grid is the Cartesian product of the 1-D scan
positions.


In [2]:
p = IndexerParams(
    px=200.0, Distance=1e6, Wavelength=0.18,
    SpaceGroup=225,
    EtaBinSize=0.1, OmeBinSize=0.1,
    StepsizeOrient=0.5,
    MarginEta=2.0, MarginOme=0.5, MarginRad=10.0, MarginRadial=10.0,
    RingNumbers=[1],
    RingRadii={1: 30000.0},
    scan_pos_tol_um=2.0,
    friedel_symmetric_scan_filter=True,
    multi_solution_output=True,
)
ind = Indexer(p, device="cpu")

# Minimal 10-column PF observations (one dummy spot, scan_nr=0) so
# load_observations() doesn't need on-disk binaries.
ind._observations = {
    "spots":     np.zeros((1, 10), dtype=np.float64),
    "bin_data":  np.zeros(0, dtype=np.int32),
    "bin_ndata": np.zeros(0, dtype=np.int32),
    "hkls_real": np.zeros((1, 6), dtype=np.float64),
    "hkls_int":  np.zeros((1, 4), dtype=np.int64),
    "spot_ids":  np.zeros(0, dtype=np.int64),
}
print("scan filter tol (µm):", p.scan_pos_tol_um,
      "| friedel-symmetric:", p.friedel_symmetric_scan_filter)


scan filter tol (µm): 2.0 | friedel-symmetric: True


## 2. Run the scanning indexer

`run_scanning` iterates the `n_scans × n_scans` voxel grid built from
`scan_positions` and writes the consolidated `IndexBest_all.bin`. It
returns the number of voxels processed. (Large grids can be sharded
with `voxel_block_nr` / `voxel_n_blocks`.)


In [3]:
scan_positions = np.array([0.0, 5.0, 10.0])     # 3 positions → 3×3 = 9 voxels
out_path = WORK / "IndexBest_all.bin"

n_vox = ind.run_scanning(
    scan_positions=scan_positions,
    out_path=out_path,
    num_procs=1,
    backend="python",
)
print("voxels processed:", n_vox)
print("wrote:", out_path.name, f"({out_path.stat().st_size} bytes)")


voxels processed: 9
wrote: IndexBest_all.bin (112 bytes)


## 3. Read back the consolidated output

`read_index_best_all` parses the consolidated binary: a header, a
per-voxel solution-count array, and the records. With no real
observations every voxel has zero solutions — but the byte layout,
voxel count, and reader round-trip are exactly what a production
scanning run produces.


In [4]:
res = read_index_best_all(out_path)
print("n_voxels in file :", res.n_voxels)
print("solutions/voxel  :", res.n_sol_arr.tolist())
print()
print("Voxel grid (Cartesian product of scan positions, µm):")
n = len(scan_positions)
for i in range(n):
    for j in range(n):
        v = i * n + j
        print(f"  voxel {v}: (x={scan_positions[j]:5.1f}, y={scan_positions[i]:5.1f})"
              f"  n_solutions={int(res.n_sol_arr[v])}")


n_voxels in file : 9
solutions/voxel  : [0, 0, 0, 0, 0, 0, 0, 0, 0]

Voxel grid (Cartesian product of scan positions, µm):
  voxel 0: (x=  0.0, y=  0.0)  n_solutions=0
  voxel 1: (x=  5.0, y=  0.0)  n_solutions=0
  voxel 2: (x= 10.0, y=  0.0)  n_solutions=0
  voxel 3: (x=  0.0, y=  5.0)  n_solutions=0
  voxel 4: (x=  5.0, y=  5.0)  n_solutions=0
  voxel 5: (x= 10.0, y=  5.0)  n_solutions=0
  voxel 6: (x=  0.0, y= 10.0)  n_solutions=0
  voxel 7: (x=  5.0, y= 10.0)  n_solutions=0
  voxel 8: (x= 10.0, y= 10.0)  n_solutions=0


## Recap

- One indexer, two modes: FF is the `n_scans=1` specialization of the
  scanning algorithm; PF activates the per-voxel scan-position filter.
- `Indexer.run_scanning(scan_positions, out_path=…)` writes the
  consolidated `IndexBest_all.bin` over the voxel grid.
- `read_index_best_all` round-trips it.
- For real PF data, point `load_observations(cwd=…)` at a directory
  with 10-column `Spots.bin` + scanning `Data.bin`/`nData.bin`, and
  pass the real `positions.csv` values as `scan_positions`. The
  `c-omp` backend handles production-scale grids.
